# QST-STOR-0001: SNS Storage Geometry Audit

This notebook reproduces the geometry-first audit for the SNS seed battery. It treats storage as a survival buffer, not as a warehouse for the full PV kite output.

The executable source of truth lives in `src/sim/storage_geometry.py`; this notebook is the human-facing lens.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from experiments.storage_geometry_audit import run_storage_geometry_audit

OUTPUT = ROOT / 'outputs' / 'qst_stor_0001'
rows, summary = run_storage_geometry_audit(
    ROOT / 'configs' / 'storage_geometry_audit.json', OUTPUT
)
summary['scenario_count'], summary['pass_count'], summary['fail_count']


## Capacity and pass rate by core diameter

PASS is a geometry gate: usable capacity covers the selected shadow interval after the configured discharge loss and reserve. It is not flight qualification.


In [ ]:
for core, values in summary['by_core_diameter'].items():
    print(
        core,
        f"pass={values['pass_rate']:.1%}",
        f"capacity={values['usable_battery_Wh_min']:.6f}–{values['usable_battery_Wh_max']:.6f} Wh",
    )


## Attack the baseline assumption

The nominal sweep assumes 1% active duty during shadow. The sensitivity table shows how quickly storage feasibility erodes when the node tries to remain active in darkness.


In [ ]:
for stress in summary['active_duty_cycle_sensitivity']:
    print(
        f"duty={stress['active_duty_cycle']:.1%}",
        f"pass={stress['pass_rate']:.1%}",
    )


## Inspect the failure frontier


In [ ]:
failures = [row for row in rows if row['status'] == 'FAIL']
worst = min(rows, key=lambda row: row['storage_margin_Wh'])
len(failures), worst


## Interpretation

1. The 10 mm core is not impossible as a survival-buffer geometry, but it is highly sensitive to long shadows and active duty.
2. The canonical 0.1–10 Wh range should be replaced by geometry-derived capacity ranges.
3. `pv_fill_time_s` is only an energy-flow lower bound. The 1C-limited result demonstrates that electrochemical acceptance, not sunlight availability, sets the charging clock.
4. The next model should add temperature-dependent capacity, heater load, degradation, and dedicated host/storage nodes.
